# BinSense — M3b · Track 2 variant: **M4 detector as the second opinion**

The `SAM ∩ zero-shot` A/B (nb `03c`) failed: zero-shot was near-blind on these
cluttered bins, so consensus starved (7/202 bins, 9 mostly-wrong boxes on
dividers/tape). This notebook tests the salvage hypothesis: **your M4 detector,
though weak (mAP 0.255), was trained on *these exact items*, so it should be a far
better cross-check for SAM than a generic open-vocab model.**

Only one variable changes vs `03c`: the second method is now **M4**, not zero-shot.
Same SAM boxes, same `GateParams`, same engine (`cross_method_agreement` is generic).
Watch for the *correlated-error* risk — M4 also over-fires on texture, so it may
agree with SAM on the wrong things. The overlays + the head-to-head decide it.

In [ ]:
# Cell 1: Bootstrap — path resolution + data/code split
import sys, os, subprocess
from pathlib import Path

GITHUB_URL = 'https://github.com/rishib09/AmazonBinSense.git'
BRANCH     = 'm3b-auto-labeling'   # set 'master' after this milestone merges
DRIVE_ROOT = '/content/drive/MyDrive/Interview Kickstart/Capstone Project/Amazon BinSense'
LOCAL_DATA = r'G:\My Drive\Interview Kickstart\Capstone Project\Amazon BinSense\data'

try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/AmazonBinSense')
    if (PROJECT_ROOT / '.git').exists():
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', BRANCH], check=False)
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', BRANCH], check=False)
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', 'origin', BRANCH, '--ff-only'], check=False)
    else:
        subprocess.run(['git', 'clone', '--branch', BRANCH, GITHUB_URL, str(PROJECT_ROOT)], check=True)
    os.environ['BINSENSE_DATA_DIR'] = str(Path(DRIVE_ROOT) / 'data')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python-headless', 'pandas', 'pyyaml', 'matplotlib'], check=True)
except ImportError:
    IN_COLAB = False
    if os.getenv('BINSENSE_DIR'):
        PROJECT_ROOT = Path(os.environ['BINSENSE_DIR'])
    else:
        _cwd = Path.cwd()
        PROJECT_ROOT = _cwd.parent if _cwd.name == 'notebooks' else _cwd
    if not os.getenv('BINSENSE_DATA_DIR') and Path(LOCAL_DATA).exists():
        os.environ['BINSENSE_DATA_DIR'] = LOCAL_DATA

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Running in:', 'Google Colab' if IN_COLAB else 'Local', '| ROOT:', PROJECT_ROOT)

In [ ]:
# Cell 2: Imports + paths — raw method dirs, track output dir, eval wall
import json
import pandas as pd
from pathlib import Path
from utils.env_utils import setup_env

cfg = setup_env(verbose=True)
IMAGES_DIR = cfg.images_dir
META_DIR   = cfg.metadata_dir
SEED_DIR   = cfg.labels_dir                       # 120-bin manual seed (ground truth)

# Raw per-method boxes (written by the GPU cells below; persist on Drive)
SAM_DIR = cfg.data_dir / 'labels_auto' / 'sam'
ZS_DIR  = cfg.data_dir / 'labels_auto' / 'zeroshot'
SAM_DIR.mkdir(parents=True, exist_ok=True)
ZS_DIR.mkdir(parents=True, exist_ok=True)

# The one wall we never cross: eval-gold bins are the M7 test set.
EVAL_IDS = set(pd.read_csv(cfg.splits_dir / 'eval.csv')['bin_id'].astype(str).str.zfill(5))
EXTEND   = pd.read_csv(cfg.splits_dir / 'extend.csv')['bin_id'].astype(str).str.zfill(5).tolist()
EXTEND   = [b for b in EXTEND if b not in EVAL_IDS]
print(f'extend bins to auto-label: {len(EXTEND)}   eval held out: {len(EVAL_IDS)}   seed labels: {len(list(SEED_DIR.glob("*.txt")))}')

## Step 1 — Run M4 on the SAM-covered bins
Load `best.pt` and detect at conf 0.25 (item-ish proposals, not the low-conf flood). Writes one box file per bin to `data/labels_auto/m4/`.

In [ ]:
# Cell 3: Run the M4 detector on SAM-covered bins -> M4_DIR (the second opinion)
from tools.labeling.autolabel import write_boxes

RUN_M4     = False        # flip True on Colab (GPU ideal; CPU works, slower)
M4_CONF    = 0.25         # item-ish proposals, NOT the low-conf flood
M4_WEIGHTS = cfg.models_dir / 'yolov8s_item' / 'weights' / 'best.pt'
M4_DIR     = cfg.data_dir / 'labels_auto' / 'm4'
M4_DIR.mkdir(parents=True, exist_ok=True)

if RUN_M4:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)
    from ultralytics import YOLO
    from tqdm.auto import tqdm
    assert M4_WEIGHTS.exists(), f'M4 weights not found: {M4_WEIGHTS}'
    model = YOLO(str(M4_WEIGHTS))
    todo = [p.stem for p in SAM_DIR.glob('*.txt') if not (M4_DIR / f'{p.stem}.txt').exists()]
    done = 0
    for bid in tqdm(todo, desc='M4 detect', unit='bin'):
        r = model.predict(str(IMAGES_DIR / f'{bid}.jpg'), conf=M4_CONF, verbose=False)[0]
        confs = r.boxes.conf.tolist()
        boxes = [{'cx': cx, 'cy': cy, 'w': w, 'h': h, 'score': float(s), 'source': 'm4'}
                 for (cx, cy, w, h), s in zip(r.boxes.xywhn.tolist(), confs)]
        write_boxes(M4_DIR / f'{bid}.txt', boxes, with_score=True)
        done += 1
    print(f'M4 wrote {done} bins -> {M4_DIR}  (total on disk: {len(list(M4_DIR.glob("*.txt")))})')
else:
    print(f'RUN_M4=False. Existing M4 box files: {len(list(M4_DIR.glob("*.txt")))}')

## Step 2 — Engine sanity check (no GPU)

In [ ]:
# Cell 5: Engine sanity check (no GPU) — proves the geometry/agreement/NMS logic
from tools.labeling.autolabel import iou, nms, gate_geometry, cross_method_agreement, gate_count, GateParams

def _b(cx, cy, w, h, s=1.0):
    return {'cx': cx, 'cy': cy, 'w': w, 'h': h, 'score': s, 'source': 't'}

p = GateParams()
assert iou(_b(.5,.5,.2,.2), _b(.5,.5,.2,.2)) == 1.0
assert iou(_b(.5,.5,.2,.2), _b(.9,.9,.2,.2)) == 0.0
# geometry drops speck / full-frame / sliver, keeps a normal box
kept = gate_geometry([_b(.5,.5,.01,.01), _b(.5,.5,.99,.99), _b(.5,.5,.6,.01), _b(.5,.5,.1,.1)], p)
assert len(kept) == 1
# NMS collapses a near-duplicate
assert len(nms([_b(.5,.5,.2,.2,.9), _b(.51,.51,.2,.2,.5), _b(.1,.1,.1,.1,.8)], p.nms_iou)) == 2
# agreement keeps only the box both methods found
assert len(cross_method_agreement([_b(.3,.3,.2,.2), _b(.8,.8,.1,.1)], [_b(.31,.31,.2,.2)], p.agree_iou)) == 1
# count gate: 0 and gross over-seg rejected; visible<expected accepted
assert gate_count(0, 10, p)[0] is False and gate_count(30, 10, p)[0] is False and gate_count(8, 10, p)[0] is True
print('Engine sanity check PASSED — geometry, NMS, cross-method agreement, count gate all behave.')

## Step 3 — Build Track 2 (SAM ∩ M4)
Same gates as `03c`; M4 replaces zero-shot as the agreement partner.

In [ ]:
# Cell 6: Build Track 2 with M4 as the second opinion (SAM ∩ M4)
from tools.labeling.autolabel import build_track, GateParams

OUT2M4 = cfg.data_dir / 'labels_track2_m4'
# only bins where BOTH SAM and M4 produced boxes (the gate needs two opinions)
LABELED = sorted({p.stem for p in SAM_DIR.glob('*.txt')} & {p.stem for p in M4_DIR.glob('*.txt')})
print('bins with both SAM and M4 boxes:', len(LABELED))

p = GateParams()   # SAME params as 03c so the ONLY changed variable is the 2nd method
# M4 boxes occupy the engine's second-method (zs) slot; cross_method_agreement is generic,
# so this computes SAM ∩ M4. Manifest columns n_zs / n_zs_geom therefore mean M4 here.
st_m4 = build_track('track2', SAM_DIR, M4_DIR, OUT2M4, META_DIR, LABELED, EVAL_IDS, p)
print('Track 2 (SAM ∩ M4):', json.dumps(st_m4.as_dict(), indent=2))

man_m4 = pd.DataFrame(st_m4.manifest)
if len(man_m4):
    print(f"accepted: {st_m4.n_bins_out}   rejected: {st_m4.n_rejected}   boxes kept: {st_m4.n_boxes_out}")
    print('reject reasons:', st_m4.reject_reasons)
    display(man_m4.rename(columns={'n_zs': 'n_m4', 'n_zs_geom': 'n_m4_geom'}).head(12))
else:
    print('No results — run Cell 3 (M4 detect) and 03b Step 1 (SAM) first.')

## Step 4 — Gate funnel + survivor overlays
Does the consensus bar survive this time? And do the kept boxes hug real items?

In [ ]:
# Cell 7: Gate funnel (SAM ∩ M4) + overlays of the survivors
import matplotlib.pyplot as plt, cv2, numpy as np
from tools.labeling.overlay_check import draw_overlay, parse_label

if len(man_m4) and 'n_consensus' in man_m4:
    funnel = {
        'raw (sam+m4)':   int((man_m4['n_sam'] + man_m4['n_zs']).sum()),
        'geometry-kept':  int((man_m4['n_sam_geom'] + man_m4['n_zs_geom']).sum()),
        'consensus SAM∩M4': int(man_m4['n_consensus'].sum()),
        'final (Gate A)': int(man_m4['n_final'].sum()),
    }
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].bar(range(len(funnel)), list(funnel.values()), color='teal')
    ax[0].set_xticks(range(len(funnel))); ax[0].set_xticklabels(list(funnel), rotation=20, ha='right')
    ax[0].set_title('box funnel — M4 as second opinion'); ax[0].set_ylabel('total boxes')
    for i, v in enumerate(funnel.values()):
        ax[0].text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=9)
    reasons = st_m4.reject_reasons
    if reasons:
        ax[1].bar(list(reasons), list(reasons.values()), color='indianred')
        ax[1].set_title('bin reject reasons'); ax[1].set_ylabel('#bins')
    plt.tight_layout(); plt.show()

    shown = list(man_m4[man_m4['passed']]['bin_id'])[:3]
    if shown:
        fig, axes = plt.subplots(1, len(shown), figsize=(5*len(shown), 5))
        for ax_, bid in zip(np.atleast_1d(axes), shown):
            out = cfg.base_dir / 'reports' / 'track2_m4_overlays' / f'{bid}.jpg'
            draw_overlay(IMAGES_DIR/f'{bid}.jpg', parse_label(OUT2M4/f'{bid}.txt'), out)
            ax_.imshow(cv2.cvtColor(cv2.imread(str(out)), cv2.COLOR_BGR2RGB)); ax_.set_title(bid); ax_.axis('off')
        plt.suptitle('Track 2 survivors — SAM ∩ M4'); plt.show()
else:
    print('Nothing to visualize yet.')

## Step 5 — Head-to-head decision
Directly compare the two second-opinion choices on the identical SAM boxes.

In [ ]:
# Cell 8: Head-to-head — which second opinion salvages consensus?
from tools.labeling.autolabel import read_boxes

def summarize_dir(d):
    files = list(Path(d).glob('*.txt'))
    nb = sum(len(read_boxes(f)) for f in files)
    return len(files), nb

zs_bins, zs_boxes = summarize_dir(cfg.data_dir / 'labels_track2')       # SAM ∩ zero-shot (03c)
m4_bins, m4_boxes = summarize_dir(OUT2M4)                               # SAM ∩ M4 (this nb)
print('Second-opinion comparison on the same SAM boxes:')
print(f'  SAM ∩ zero-shot : {zs_bins:>4} bins accepted, {zs_boxes:>5} boxes kept')
print(f'  SAM ∩ M4        : {m4_bins:>4} bins accepted, {m4_boxes:>5} boxes kept')
print('\nDecision: if SAM ∩ M4 accepts materially more bins AND the survivor overlays')
print('look like real items (not dividers/tape), M4 salvages the pipeline -> scale it.')
print('If it is still sparse or the boxes are wrong, auto-labeling is not viable here')
print('and the clean manual-seed expansion (130 -> 250) is the path.')

## Verdict → next step

- **M4 salvages it** (many more bins accepted, survivor boxes look like items): scale
  SAM + M4 over several hundred `extend` bins, merge the gated labels with your growing
  manual seed, and retrain M4 on the union.
- **M4 also fails** (still sparse or boxes on dividers/tape): the honest finding is that
  auto-labeling is not viable on ABID bins; rely on the clean manual-seed expansion
  (130 → 250) and record the negative result for the Level-3 deep-dive.

Either outcome is a legitimate, defensible result — that's what the A/B was for.